# EDA — Synthetic Arrhenius Dataset

This notebook performs quick sanity checks and visualisations for `data/dataset.csv`.

It attempts several common relative paths (useful when opening from Colab or from the `notebooks/` folder).

Outputs:
- head() and summary statistics
- histogram of reaction rate `k`
- scatter plot `k` vs `T`
- boxplot of `k` by `catalyst`
- correlation matrix (numeric)

In [ ]:
# Try to load dataset from common relative paths. Prints which path worked.
import os
import pandas as pd

candidates = [
    '../data/dataset.csv',   # if notebook is in notebooks/ and repo root has data/
    './data/dataset.csv',    # if notebook run from repo root
    '/content/data/dataset.csv',
    'data/dataset.csv',
    '../data/dataset.csv'
]

df = None
for path in candidates:
    if os.path.exists(path):
        try:
            df = pd.read_csv(path)
            print(f'Loaded dataset from: {path}')
            break
        except Exception as e:
            print(f'Found file at {path} but failed to read: {e}')

if df is None:
    raise FileNotFoundError('\nCould not find data/dataset.csv in common locations.\nPlease upload dataset.csv to your repo data/ folder or adjust the path.')

print('\nDataset shape:', df.shape)
df.head()

In [ ]:
# Basic summary statistics
df.describe(include='all')

In [ ]:
# Import plotting libs
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (8,5)
sns.set(style='whitegrid')

# Histogram of k (reaction rate)
plt.figure()
sns.histplot(df['k'].dropna(), bins=50, kde=False)
plt.xlabel('Reaction rate k')
plt.title('Distribution of reaction rate (k)')
plt.show()

In [ ]:
# Scatter: k vs T (full dataset)
plt.figure()
plt.scatter(df['T'], df['k'], s=6, alpha=0.6)
plt.xlabel('Temperature (K)')
plt.ylabel('Reaction rate k')
plt.title('k vs Temperature')
plt.show()

# Optional: clearer view for a restricted Ea range
if 'Ea' in df.columns:
    subset = df[(df['Ea'] > df['Ea'].quantile(0.25)) & (df['Ea'] < df['Ea'].quantile(0.75))]
    plt.figure()
    plt.scatter(subset['T'], subset['k'], s=6, alpha=0.6)
    plt.xlabel('Temperature (K)')
    plt.ylabel('Reaction rate k')
    plt.title('k vs T (middle 50% Ea)')
    plt.show()

In [ ]:
# Boxplot to check catalyst effect (if present)
if 'catalyst' in df.columns:
    plt.figure()
    sns.boxplot(x='catalyst', y='k', data=df)
    plt.xlabel('Catalyst (0 = no, 1 = yes)')
    plt.ylabel('Reaction rate k')
    plt.title('Effect of catalyst on k')
    plt.show()
else:
    print('No catalyst column found; skipping catalyst boxplot.')

In [ ]:
# Correlations (numeric columns)
num = df.select_dtypes(include=['number']).copy()
if not num.empty:
    display(num.corr())
else:
    print('No numeric columns to correlate.')

### Save figures (optional)

If you want to save the main figures to the repo `assets/` folder (so you can include them in the poster), create the folder and run the cells below. In Colab you can then download the PNGs or push them back to GitHub manually.

In [ ]:
import os
os.makedirs('../assets', exist_ok=True)
fig_path = '../assets/k_hist.png'
plt.figure()
sns.histplot(df['k'].dropna(), bins=50)
plt.xlabel('Reaction rate k')
plt.title('Distribution of k')
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
print('Saved histogram to', fig_path)